In [1]:
# A. Membangun Catalog (Create)
from pymongo import MongoClient

# Koneksi ke MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
koleksi_menu = db["menu"]

# Bersihkan collection jika ada data lama
koleksi_menu.delete_many({})

# Menyisipkan minimal 12 dokumen dengan field bervariasi
koleksi_menu.insert_many([
    {"nama_menu": "Espresso", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "rekomendasi_barista": True},
    {"nama_menu": "Americano", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 4, "rekomendasi_barista": False},
    {"nama_menu": "Cafe Latte", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 2, "rekomendasi_barista": True},
    {"nama_menu": "Cappuccino", "harga": 27000, "kategori": "Kopi", "tingkat_kepahitan": 3, "rekomendasi_barista": True},
    {"nama_menu": "Mochaccino", "harga": 30000, "kategori": "Kopi", "tingkat_kepahitan": 3, "rekomendasi_barista": False},

    {"nama_menu": "Nasi Goreng Special", "harga": 28000, "kategori": "Makanan", "pedas_level": 2, "stok": 20},
    {"nama_menu": "Mie Goreng Nusantara", "harga": 22000, "kategori": "Makanan", "pedas_level": 3, "stok": 15},
    {"nama_menu": "Roti Bakar Coklat", "harga": 15000, "kategori": "Makanan", "pedas_level": 0, "stok": 25},
    {"nama_menu": "Kentang Goreng", "harga": 18000, "kategori": "Makanan", "pedas_level": 1, "stok": 30},
    {"nama_menu": "Pisang Goreng Keju", "harga": 14000, "kategori": "Makanan", "pedas_level": 0, "stok": 25},

    {"nama_menu": "Es Teh Manis", "harga": 8000, "kategori": "Minuman", "ukuran": "Large", "stok": 50},
    {"nama_menu": "Jus Alpukat", "harga": 22000, "kategori": "Minuman", "ukuran": "Regular", "stok": 20}
])

print("Total dokumen menu berhasil disisipkan:", koleksi_menu.count_documents({}))

Total dokumen menu berhasil disisipkan: 12


In [2]:
# B. Kueri Bertingkat (Read)
# 1. Seluruh menu kategori "Kopi" dengan tingkat_kepahitan di atas 3
print("1. Kopi dengan tingkat kepahitan > 3:")
for m in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f"   - {m['nama_menu']} (Kepahitan: {m['tingkat_kepahitan']})")

print()

# 2. Seluruh menu dengan harga di antara 15000 dan 30000
print("2. Menu dengan harga antara Rp15.000 - Rp30.000:")
for m in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f"   - {m['nama_menu']} (Rp{m['harga']:,})")

print()

# 3. Menu yang TIDAK MEMILIKI field pedas_level sama sekali
print("3. Menu yang tidak memiliki field 'pedas_level':")
for m in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"   - {m['nama_menu']} ({m['kategori']})")

1. Kopi dengan tingkat kepahitan > 3:
   - Espresso (Kepahitan: 5)
   - Americano (Kepahitan: 4)

2. Menu dengan harga antara Rp15.000 - Rp30.000:
   - Espresso (Rp18,000)
   - Americano (Rp20,000)
   - Cafe Latte (Rp25,000)
   - Cappuccino (Rp27,000)
   - Mochaccino (Rp30,000)
   - Nasi Goreng Special (Rp28,000)
   - Mie Goreng Nusantara (Rp22,000)
   - Roti Bakar Coklat (Rp15,000)
   - Kentang Goreng (Rp18,000)
   - Jus Alpukat (Rp22,000)

3. Menu yang tidak memiliki field 'pedas_level':
   - Espresso (Kopi)
   - Americano (Kopi)
   - Cafe Latte (Kopi)
   - Cappuccino (Kopi)
   - Mochaccino (Kopi)
   - Es Teh Manis (Minuman)
   - Jus Alpukat (Minuman)


In [3]:
# C. Pembaruan Data (Update)
# Tambahkan field 'status' bernilai "Tersedia" ke seluruh dokumen
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})

# Ubah 'status' menjadi "Habis" khusus untuk 2 menu pilihan
koleksi_menu.update_one({"nama_menu": "Espresso"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Nasi Goreng Special"}, {"$set": {"status": "Habis"}})

print("Status berhasil diperbarui!")
for m in koleksi_menu.find({}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(f"  - {m['nama_menu']}: {m['status']}")

Status berhasil diperbarui!
  - Espresso: Habis
  - Americano: Tersedia
  - Cafe Latte: Tersedia
  - Cappuccino: Tersedia
  - Mochaccino: Tersedia
  - Nasi Goreng Special: Habis
  - Mie Goreng Nusantara: Tersedia
  - Roti Bakar Coklat: Tersedia
  - Kentang Goreng: Tersedia
  - Pisang Goreng Keju: Tersedia
  - Es Teh Manis: Tersedia
  - Jus Alpukat: Tersedia


In [6]:
# D. Integrasi ke Spark

from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql.functions import avg

spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 1. Ambil data dari MongoDB dan ubah _id ObjectId menjadi string
data_menu = list(koleksi_menu.find({}))
for doc in data_menu:
    doc["_id"] = str(doc["_id"])

# 2. Ubah ke pandas DataFrame lalu ekspor ke format JSON string per baris
df_pd = pd.DataFrame(data_menu)
json_data = df_pd.to_json(orient="records", lines=True)

# 3. Baca menggunakan Spark JSON Reader (sangat aman untuk struktur dokumen yang bervariasi)
df_spark = spark.read.json(spark.sparkContext.parallelize(json_data.splitlines()))

print("\n--- SKEMA SPARK DATAFRAME ---")
df_spark.printSchema()

print("--- RATA-RATA HARGA PER KATEGORI ---")
df_spark.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga")
).show()

spark.stop()
print("SparkSession ditutup.")


--- SKEMA SPARK DATAFRAME ---
root
 |-- _id: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- pedas_level: double (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)
 |-- status: string (nullable = true)
 |-- stok: double (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- ukuran: string (nullable = true)

--- RATA-RATA HARGA PER KATEGORI ---


+--------+---------------+
|kategori|rata_rata_harga|
+--------+---------------+
|    Kopi|        24000.0|
| Makanan|        19400.0|
| Minuman|        15000.0|
+--------+---------------+

SparkSession ditutup.


# E. Kesimpulan Refleksi

Tantangan utama saat mengonversi data dari basis data NoSQL seperti MongoDB ke dalam Spark DataFrame adalah sifat dokumen NoSQL yang fleksibel (flexible schema), di mana setiap dokumen dapat memiliki atribut yang tidak seragam (misalnya menu Kopi memiliki atribut tingkat_kepahitan sedangkan menu Makanan memiliki atribut pedas_level). Akibatnya, ketika diubah melalui Pandas menjadi Spark DataFrame, kolom-kolom yang tidak ada pada dokumen tertentu akan otomatis diisi dengan nilai kosong (null) dan tipe data kolom perlu disesuaikan dengan hati-hati (seperti mengubah tipe objek _id menjadi string). Untuk mengatasinya, saya memastikan struktur data dasar seperti nama_menu, harga, dan kategori terstandarisasi dengan baik pada setiap dokumen, sementara atribut spesifik dibiarkan bernilai null pada kategori lain tanpa mengganggu proses agregasi seperti groupBy di Spark.